In [1]:
import pandas as pd
from utils.read import readExcel
from datetime import datetime

In [ ]:
df = readExcel(
    "./raw-sheets-dump/17th July 2026.xlsx",
    "Sheet1",
)[0]

shelfLife = readExcel("./shelf-life/Summer 2026 PUNE City Shelf Life.xlsx", "Sheet1")[0]
shelfLife.columns = shelfLife.iloc[0]
shelfLife = shelfLife.iloc[1:]

nx = readExcel("./shelf-life/NX Article SL.xlsx", "Sheet1")[0]

df.columns = df.columns.str.strip()
shelfLife.columns = shelfLife.columns.str.strip()
nx.columns = nx.columns.str.strip()

In [3]:
merged = df.merge(shelfLife, how="inner", left_on="ITEM_CODE", right_on="Article Code")

In [4]:
finalDf = merged.copy()
finalDf.columns = finalDf.columns.str.strip()
finalDf = finalDf[["PRODUCT_NAME", "WEIGHT", "Symbol", "Date", "ITEM_CODE", "Indents"]]
finalDf

,PRODUCT_NAME,WEIGHT,Symbol,Date,ITEM_CODE,Indents
0,Ash Gourd (Cut Portion),200g,C,2026-07-17,9009.0,1.0
1,Ash Gourd Portion,1Piece,C,2026-07-17,5949.0,1.0
2,Ash Gourd Portion,1Piece,C,2026-07-17,5949.0,1.0
3,Ash Gourd Portion,1Piece,C,2026-07-17,5949.0,1.0
4,Baby Banana,4Pieces,T,2026-07-17,11565.0,31.0
...,...,...,...,...,...,...
524,Yellow Shevanti Flowers,100g,C,2026-07-17,9497.0,3.0
525,Yellow Shevanti Flowers,100g,C,2026-07-17,9497.0,2.0
526,Yellow Shevanti Flowers,100g,C,2026-07-17,9497.0,3.0
527,Yellow Shevanti Flowers,100g,C,2026-07-17,9497.0,2.0


In [ ]:
finalDf["Date"] = pd.to_datetime(finalDf["Date"])
finalDf["Date"] = finalDf["Date"] - pd.Timedelta(days=1)

In [6]:
day = finalDf["Date"].apply(lambda x: (x.isoweekday() % 7) + 1).unique()[0] - 1
dayOfTheWeek = shelfLife.columns[8:15][day]

In [7]:
finalDf[dayOfTheWeek] = finalDf.merge(
    shelfLife, how="inner", left_on="ITEM_CODE", right_on="Article Code"
)[dayOfTheWeek]

In [14]:
finalDf["ITEM_CODE"] = finalDf["ITEM_CODE"].astype(int)
nx["ITEM_CODE"] = nx["ITEM_CODE"].astype(int)
finalDf["Day"] = finalDf[dayOfTheWeek].astype(int)

In [ ]:
finalDf["ID"] = finalDf.apply(
    lambda row: "b_"
    + str(row["ITEM_CODE"])
    + "_"
    + (row["Date"] + pd.Timedelta(days=1)).strftime("%d-%m-%Y"),
    axis=1,
)

In [22]:
finalDf["NX"] = ""
for i in nx["ITEM_CODE"]:
    finalDf.loc[finalDf["ITEM_CODE"] == i, "NX"] = "NX"

In [ ]:
finalDf = finalDf.loc[finalDf.index.repeat(finalDf["Indents"])]

,PRODUCT_NAME,WEIGHT,Symbol,Date,ITEM_CODE,Indents,Friday,Day,ID
0,Ash Gourd (Cut Portion),200g,C,2026-07-17,9009,1.0,7.0,7,b_9009_17-07-2026
1,Ash Gourd Portion,1Piece,C,2026-07-17,5949,1.0,7.0,7,b_5949_17-07-2026
2,Ash Gourd Portion,1Piece,C,2026-07-17,5949,1.0,7.0,7,b_5949_17-07-2026
3,Ash Gourd Portion,1Piece,C,2026-07-17,5949,1.0,7.0,7,b_5949_17-07-2026
4,Baby Banana,4Pieces,T,2026-07-17,11565,31.0,1.0,1,b_11565_17-07-2026
...,...,...,...,...,...,...,...,...,...
527,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,2.0,7.0,7,b_9497_17-07-2026
527,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,2.0,7.0,7,b_9497_17-07-2026
528,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,3.0,7.0,7,b_9497_17-07-2026
528,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,3.0,7.0,7,b_9497_17-07-2026


In [23]:
finalDf

,PRODUCT_NAME,WEIGHT,Symbol,Date,ITEM_CODE,Indents,Friday,Day,ID,NX
0,Ash Gourd (Cut Portion),200g,C,2026-07-17,9009,1.0,7.0,7,b_9009_17-07-2026,
1,Ash Gourd Portion,1Piece,C,2026-07-17,5949,1.0,7.0,7,b_5949_17-07-2026,
2,Ash Gourd Portion,1Piece,C,2026-07-17,5949,1.0,7.0,7,b_5949_17-07-2026,
3,Ash Gourd Portion,1Piece,C,2026-07-17,5949,1.0,7.0,7,b_5949_17-07-2026,
4,Baby Banana,4Pieces,T,2026-07-17,11565,31.0,1.0,1,b_11565_17-07-2026,NX
...,...,...,...,...,...,...,...,...,...,...
524,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,3.0,7.0,7,b_9497_17-07-2026,
525,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,2.0,7.0,7,b_9497_17-07-2026,
526,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,3.0,7.0,7,b_9497_17-07-2026,
527,Yellow Shevanti Flowers,100g,C,2026-07-17,9497,2.0,7.0,7,b_9497_17-07-2026,
